In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("AttendanceAnalysis") \
    .getOrCreate()

print("Spark session created successfully")

Spark session created successfully


In [3]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_attendance.csv to cleaned_attendance.csv
Saving employee_performance_report.csv to employee_performance_report.csv
Saving productivity_report.csv to productivity_report.csv
Saving attendance.csv to attendance.csv
Saving attendance_analysis.py to attendance_analysis.py
Saving tasks.csv to tasks.csv


In [4]:
attendance_df = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

tasks_df = spark.read.csv(
    "tasks.csv",
    header=True,
    inferSchema=True
)

print("Attendance Data")
attendance_df.show(5)

print("Task Data")
tasks_df.show(5)

Attendance Data
+-----------+-------------+----------+---------------+-------------------+-------------------+-------------+-----------------+
|employee_id|employee_name|department|attendance_date|           clock_in|          clock_out|break_minutes|attendance_status|
+-----------+-------------+----------+---------------+-------------------+-------------------+-------------+-----------------+
|          1|         Arun|        IT|     2026-10-12|2026-10-27 09:00:00|2026-10-27 17:00:00|           60|          Present|
|          2|        Priya|        HR|     2026-10-12|2026-10-27 09:15:00|2026-10-27 17:15:00|           45|          Present|
|          3|        Rahul|   Finance|     2026-10-12|               NULL|               NULL|            0|           Absent|
|          4|        Meena|        IT|     2026-10-12|2026-10-27 09:05:00|2026-10-27 17:05:00|           60|          Present|
|          1|         Arun|        IT|     2026-10-13|2026-10-27 09:10:00|2026-10-27 17:10:00| 

In [5]:
attendance_df.printSchema()
tasks_df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)
 |-- break_minutes: integer (nullable = true)
 |-- attendance_status: string (nullable = true)

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- tasks_assigned: integer (nullable = true)
 |-- tasks_completed: integer (nullable = true)



In [6]:
absent_df = attendance_df.filter(
    lower(col("attendance_status")) == "absent"
)

print("Absent Employees")
absent_df.show()

Absent Employees
+-----------+-------------+----------+---------------+--------+---------+-------------+-----------------+
|employee_id|employee_name|department|attendance_date|clock_in|clock_out|break_minutes|attendance_status|
+-----------+-------------+----------+---------------+--------+---------+-------------+-----------------+
|          3|        Rahul|   Finance|     2026-10-12|    NULL|     NULL|            0|           Absent|
|          3|        Rahul|   Finance|     2026-10-13|    NULL|     NULL|            0|           Absent|
|          3|        Rahul|   Finance|     2026-10-14|    NULL|     NULL|            0|           Absent|
+-----------+-------------+----------+---------------+--------+---------+-------------+-----------------+



In [7]:
late_login_df = attendance_df.filter(
    (col("clock_in").isNotNull()) &
    (date_format(col("clock_in"), "HH:mm:ss") > "09:00:00")
)

print("Late Login Records")
late_login_df.show()

Late Login Records
+-----------+-------------+----------+---------------+-------------------+-------------------+-------------+-----------------+
|employee_id|employee_name|department|attendance_date|           clock_in|          clock_out|break_minutes|attendance_status|
+-----------+-------------+----------+---------------+-------------------+-------------------+-------------+-----------------+
|          2|        Priya|        HR|     2026-10-12|2026-10-27 09:15:00|2026-10-27 17:15:00|           45|          Present|
|          4|        Meena|        IT|     2026-10-12|2026-10-27 09:05:00|2026-10-27 17:05:00|           60|          Present|
|          1|         Arun|        IT|     2026-10-13|2026-10-27 09:10:00|2026-10-27 17:10:00|           60|          Present|
|          2|        Priya|        HR|     2026-10-13|2026-10-27 09:20:00|2026-10-27 16:50:00|           45|          Present|
|          2|        Priya|        HR|     2026-10-14|2026-10-27 09:10:00|2026-10-27 17:10:0

In [8]:
attendance_hours_df = attendance_df.withColumn(
    "clock_in_time",
    to_timestamp(col("clock_in"))
).withColumn(
    "clock_out_time",
    to_timestamp(col("clock_out"))
).withColumn(
    "work_hours",
    (unix_timestamp(col("clock_out_time")) -
     unix_timestamp(col("clock_in_time"))) / 3600
)

attendance_hours_df.select(
    "employee_id",
    "department",
    "attendance_status",
    "work_hours"
).show()

+-----------+----------+-----------------+----------+
|employee_id|department|attendance_status|work_hours|
+-----------+----------+-----------------+----------+
|          1|        IT|          Present|       8.0|
|          2|        HR|          Present|       8.0|
|          3|   Finance|           Absent|      NULL|
|          4|        IT|          Present|       8.0|
|          1|        IT|          Present|       8.0|
|          2|        HR|          Present|       7.5|
|          3|   Finance|           Absent|      NULL|
|          4|        IT|          Present|       8.5|
|          1|        IT|          Present|       8.0|
|          2|        HR|          Present|       8.0|
|          3|   Finance|           Absent|      NULL|
|          4|        IT|          Present|       8.0|
+-----------+----------+-----------------+----------+



In [9]:
department_attendance_df = attendance_hours_df.groupBy(
    "department"
).agg(
    count("*").alias("total_records"),
    sum(
        when(
            lower(col("attendance_status")) == "absent",
            1
        ).otherwise(0)
    ).alias("total_absences"),
    sum(
        when(
            col("clock_in").isNotNull() &
            (date_format(col("clock_in"), "HH:mm:ss") > "09:00:00"),
            1
        ).otherwise(0)
    ).alias("late_logins"),
    round(avg("work_hours"), 2).alias("average_work_hours")
)

department_attendance_df.show()

+----------+-------------+--------------+-----------+------------------+
|department|total_records|total_absences|late_logins|average_work_hours|
+----------+-------------+--------------+-----------+------------------+
|        HR|            3|             0|          3|              7.83|
|   Finance|            3|             3|          0|              NULL|
|        IT|            6|             0|          3|              8.08|
+----------+-------------+--------------+-----------+------------------+



In [10]:
tasks_df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- tasks_assigned: integer (nullable = true)
 |-- tasks_completed: integer (nullable = true)



In [11]:
department_productivity_df = tasks_df.groupBy(
    "department"
).agg(
    sum("tasks_assigned").alias("total_tasks_assigned"),
    sum("tasks_completed").alias("total_tasks_completed")
).withColumn(
    "productivity_percentage",
    when(
        col("total_tasks_assigned") > 0,
        round(
            col("total_tasks_completed") /
            col("total_tasks_assigned") * 100,
            2
        )
    ).otherwise(0)
)

department_productivity_df.show()

+----------+--------------------+---------------------+-----------------------+
|department|total_tasks_assigned|total_tasks_completed|productivity_percentage|
+----------+--------------------+---------------------+-----------------------+
|        HR|                  10|                    8|                   80.0|
|   Finance|                   9|                    2|                  22.22|
|        IT|                  25|                   22|                   88.0|
+----------+--------------------+---------------------+-----------------------+

